In [ ]:
!pip install -q -U transformers accelerate bitsandbytes datasets huggingface_hub tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.1 MB/s eta 0:00:00


In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from huggingface_hub import login
from google.colab import userdata

# 2. Authenticate with Hugging Face
print("Authenticating...")
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

# ---> IMPORTANT: Replace with your actual username <---
repo_id = "ShravSiddhpura/cybersec-slopsquatting-crag"

# 3. Load the dataset
print(f"Pulling dataset from {repo_id}...")
dataset = load_dataset(repo_id, split="train")
all_data_patched = dataset.to_pandas().to_dict(orient="records")

# 4. Load Llama-3.1-8B in 4-bit Quantization
print("Downloading Llama-3.1-8B into the T4 GPU (This takes ~3 minutes)...")
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    token=hf_token
)

# Create a text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=600,
    temperature=0.7,
    return_full_text=False # Only returns the generated answer, not the prompt
)

# 5. The Patching Function
def rewrite_with_llama(prompt, rigid_answer):
    system_instruction = (
        "You are an AI cybersecurity expert. Rewrite the provided factual answer into a highly detailed, "
        "conversational response. It MUST be at least 3 paragraphs long. "
        "You MUST include the exact package name and CVE number provided. "
        "Make it sound exactly like a verbose AI assistant answering a complex coding question."
    )

    # Format the prompt exactly how Llama-3.1 expects it
    messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": f"User Prompt: {prompt}\nFactual Answer: {rigid_answer}"}
    ]

    formatted_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    try:
        output = pipe(formatted_prompt)
        return output[0]['generated_text'].strip()
    except Exception as e:
        print(f"Generation error: {e}")
        return None

# 6. Execute the Patch with Auto-Save Checkpoints
update_count = 0
checkpoint_interval = 50

print(f"Starting data patch. The script will auto-save to Hugging Face every {checkpoint_interval} rows.")

for item in tqdm(all_data_patched):
    if item['label'] == 0 and len(str(item['answer'])) < 600:
        new_answer = rewrite_with_llama(item['prompt'], item['answer'])

        if new_answer:
            item['answer'] = new_answer
            update_count += 1

            # --- AUTO-SAVE LOGIC ---
            if update_count % checkpoint_interval == 0:
                print(f"\n[CHECKPOINT] Saving {update_count} patched rows to Hugging Face...")
                df_temp = pd.DataFrame(all_data_patched)
                hf_temp = Dataset.from_pandas(df_temp)
                hf_temp.push_to_hub(repo_id, private=True)
                print("Checkpoint secured!")

# 7. Final Push
if update_count > 0:
    print("\nPushing final dataset back to Hugging Face Hub...")
    df = pd.DataFrame(all_data_patched)
    hf_dataset = Dataset.from_pandas(df)
    hf_dataset.push_to_hub(repo_id, private=True)
    print("SUCCESS! Data is permanently fixed.")
else:
    print("No rows needed updating!")

Authenticating...
Pulling dataset from ShravSiddhpura/cybersec-slopsquatting-crag...


README.md:   0%|          | 0.00/342 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/746k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/904 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

                              :  62%|######2   |  572kB /  919kB            

README.md:   0%|          | 0.00/342 [00:00<?, ?B/s]

 82%|████████▏ | 740/904 [1:32:50<23:54,  8.75s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Checkpoint secured!


 82%|████████▏ | 741/904 [1:33:34<31:14, 11.50s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
 82%|████████▏ | 742/904 [1:33:58<34:14, 12.68s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
 82%|████████▏ | 743/904 [1:34:42<44:42, 16.66s/it]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to t